In [1]:
from sklearn import feature_selection as fs
from sklearn import datasets
import numpy as np
import pandas as pd


# Removing features with low variance

VarianceThreshold removes features whose variance is below a chosen threshold. For binary features, Bernoulli variance is `p * (1 - p)`.


In [2]:
X = [[0, 0, 1],
     [0, 1, 0],
     [1, 0, 0],
     [0, 1, 1],
     [0, 1, 0],
     [0, 1, 1]]
X


[[0, 0, 1], [0, 1, 0], [1, 0, 0], [0, 1, 1], [0, 1, 0], [0, 1, 1]]

In [3]:
sel = fs.VarianceThreshold(threshold=0.8 * (1 - 0.8))
sel.fit_transform(X)


array([[0, 1],
       [1, 0],
       [0, 0],
       [1, 1],
       [1, 0],
       [1, 1]])

# Univariate feature selection

Univariate feature selection scores each feature independently. Common options include `SelectKBest`, `SelectPercentile`, `chi2`, `f_classif`, and `mutual_info_classif`.


In [4]:
iris = datasets.load_iris()
X = iris.data
y = iris.target
feature_names = np.array(iris.feature_names)
print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (150, 4)
y shape: (150,)


In [5]:
iris_df = pd.DataFrame(X, columns=feature_names)
iris_df["target"] = y
iris_df.head()


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


In [6]:
chi2_selector = fs.SelectKBest(score_func=fs.chi2, k=2)
X_chi2 = chi2_selector.fit_transform(X, y)
chi2_scores = pd.DataFrame({
    "feature": feature_names,
    "chi2_score": chi2_selector.scores_,
    "p_value": chi2_selector.pvalues_,
    "selected": chi2_selector.get_support(),
}).sort_values("chi2_score", ascending=False)
chi2_scores


,feature,chi2_score,p_value,selected
2,petal length (cm),116.312613,5.533972e-26,True
3,petal width (cm),67.048360,2.758250e-15,True
0,sepal length (cm),10.817821,4.476515e-03,False
1,sepal width (cm),3.710728,1.563960e-01,False


In [7]:
selected_chi2_features = feature_names[chi2_selector.get_support()]
pd.DataFrame(X_chi2[:10], columns=selected_chi2_features)


,petal length (cm),petal width (cm)
0,1.4,0.2
1,1.4,0.2
2,1.3,0.2
3,1.5,0.2
4,1.4,0.2
5,1.7,0.4
6,1.4,0.3
7,1.5,0.2
8,1.4,0.2
9,1.5,0.1


In [8]:
percentile_selector = fs.SelectPercentile(score_func=fs.chi2, percentile=50)
X_percentile = percentile_selector.fit_transform(X, y)
percentile_result = pd.DataFrame({
    "feature": feature_names,
    "selected_by_percentile": percentile_selector.get_support(),
    "score": percentile_selector.scores_,
}).sort_values("score", ascending=False)
print("New shape:", X_percentile.shape)
percentile_result


New shape: (150, 2)


,feature,selected_by_percentile,score
2,petal length (cm),True,116.312613
3,petal width (cm),True,67.048360
0,sepal length (cm),False,10.817821
1,sepal width (cm),False,3.710728


In [9]:
f_selector = fs.SelectKBest(score_func=fs.f_classif, k=2)
mi_selector = fs.SelectKBest(score_func=fs.mutual_info_classif, k=2)
X_f = f_selector.fit_transform(X, y)
X_mi = mi_selector.fit_transform(X, y)
comparison = pd.DataFrame({
    "feature": feature_names,
    "f_classif_score": f_selector.scores_,
    "f_classif_selected": f_selector.get_support(),
    "mutual_info_score": mi_selector.scores_,
    "mutual_info_selected": mi_selector.get_support(),
}).sort_values("f_classif_score", ascending=False)
comparison


,feature,f_classif_score,f_classif_selected,mutual_info_score,mutual_info_selected
2,petal length (cm),1180.161182,True,0.980695,True
3,petal width (cm),960.007147,True,0.981874,True
0,sepal length (cm),119.264502,False,0.479839,False
1,sepal width (cm),49.160040,False,0.342220,False


In [10]:
summary = pd.DataFrame({
    "method": ["VarianceThreshold", "SelectKBest chi2", "SelectPercentile chi2", "SelectKBest f_classif", "SelectKBest mutual_info"],
    "selected_features": [
        [f"x{i}" for i in sel.get_support(indices=True)],
        selected_chi2_features.tolist(),
        feature_names[percentile_selector.get_support()].tolist(),
        feature_names[f_selector.get_support()].tolist(),
        feature_names[mi_selector.get_support()].tolist(),
    ],
    "output_shape": [
        sel.transform(X=[[0, 0, 1]]).shape,
        X_chi2.shape,
        X_percentile.shape,
        X_f.shape,
        X_mi.shape,
    ],
})
summary


,method,selected_features,output_shape
0,VarianceThreshold,"[x1, x2]","(1, 2)"
1,SelectKBest chi2,"[petal length (cm), petal width (cm)]","(150, 2)"
2,SelectPercentile chi2,"[petal length (cm), petal width (cm)]","(150, 2)"
3,SelectKBest f_classif,"[petal length (cm), petal width (cm)]","(150, 2)"
4,SelectKBest mutual_info,"[petal length (cm), petal width (cm)]","(150, 2)"
